# TALLER 4
## Cargar CSV a MySql con carga inicial e incremantal

In [12]:
%pip install pymysql

import os
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()

DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')  
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')

engine = create_engine(f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}')
ddl = """
CREATE TABLE IF NOT EXISTS ventas (
  venta_id INT PRIMARY KEY,
  cliente VARCHAR(100),
  producto VARCHAR(100),
  categoria VARCHAR(100),
  precio_unitario DECIMAL(12,2),
  unidades INT,
  fecha DATE
);
"""

with engine.connect() as conn:
    conn.execute(text(ddl))
    
print("Tabla creada exitosamente.")

Note: you may need to restart the kernel to use updated packages.
Tabla creada exitosamente.



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:

# Lee CSV
df = pd.read_csv("ventas.csv")

# Casts / limpieza
df["precio_unitario"] = pd.to_numeric(df["precio_unitario"], errors="coerce")
df["unidades"] = pd.to_numeric(df["unidades"], errors="coerce")
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce").dt.date  # tipo date

# Carga inicial: reemplazar contenido de la tabla
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE ventas"))
df.to_sql("ventas", con=engine, if_exists="append", index=False, method="multi", chunksize=1000)

print(f"✅ Carga inicial completada. Registros: {len(df)}")


✅ Carga inicial completada. Registros: 20
